In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

<b><font size='3'> 문제에서 요구하는 DataFrame을 보일 때, 최상위 10개의 행만 보이시오.</font></b>

<b><font size='3'> 정답을 찾지 못하더라도, 정답을 찾는 중간 과정의 DataFrame을 보이고, 설명을 추가할 경우, 부분 점수가 부여됩니다.</font></b>

<font size='7'> <b>1.</b></font><font size='5'><span style="line-height:1.7"> <font color='blue'> inferSchema를 설정하지 않고</font>(또는 False로 설정하고), <b>fifa23.csv</b>를 DataFrame으로 로드한 후, 아래 2-5번 문제 해결을 위해 type 변경이 필요한 <font color="blue">column의 type을 변경</font>하시오.</span></font>

In [3]:
spark = SparkSession.builder.appName('01').config('spark.driver.host','localhost').getOrCreate()

In [4]:
fifa = spark.read.csv('fifa23.csv',header=True)
fifa.printSchema()

root
 |-- Known_As: string (nullable = true)
 |-- Full_Name: string (nullable = true)
 |-- Overall: string (nullable = true)
 |-- Potential: string (nullable = true)
 |-- Value: string (nullable = true)
 |-- Positions_Played: string (nullable = true)
 |-- Best_Position: string (nullable = true)
 |-- Nationality: string (nullable = true)
 |-- Image_Link: string (nullable = true)
 |-- Age: string (nullable = true)
 |-- Height: string (nullable = true)
 |-- Weight: string (nullable = true)
 |-- TotalStats: string (nullable = true)
 |-- BaseStats: string (nullable = true)
 |-- Club_Name: string (nullable = true)
 |-- Wage: string (nullable = true)
 |-- Release_Clause: string (nullable = true)
 |-- Club_Position: string (nullable = true)
 |-- Contract_Until: string (nullable = true)
 |-- Club_Jersey_Number: string (nullable = true)
 |-- Joined_On: string (nullable = true)
 |-- On_Loan: string (nullable = true)
 |-- Preferred_Foot: string (nullable = true)
 |-- Weak_Foot_Rating: string (null

In [5]:
# Overall, age, Wage,value 타입 변경
from pyspark.sql.types import *
fifa = fifa.withColumn('Overall',fifa.Overall.cast(IntegerType())).withColumn('Age',fifa.Age.cast(IntegerType())).\
withColumn('Wage',fifa.Wage.cast(IntegerType())).withColumn('Value',fifa.Value.cast(IntegerType()))

In [6]:
fifa.printSchema()

root
 |-- Known_As: string (nullable = true)
 |-- Full_Name: string (nullable = true)
 |-- Overall: integer (nullable = true)
 |-- Potential: string (nullable = true)
 |-- Value: integer (nullable = true)
 |-- Positions_Played: string (nullable = true)
 |-- Best_Position: string (nullable = true)
 |-- Nationality: string (nullable = true)
 |-- Image_Link: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Height: string (nullable = true)
 |-- Weight: string (nullable = true)
 |-- TotalStats: string (nullable = true)
 |-- BaseStats: string (nullable = true)
 |-- Club_Name: string (nullable = true)
 |-- Wage: integer (nullable = true)
 |-- Release_Clause: string (nullable = true)
 |-- Club_Position: string (nullable = true)
 |-- Contract_Until: string (nullable = true)
 |-- Club_Jersey_Number: string (nullable = true)
 |-- Joined_On: string (nullable = true)
 |-- On_Loan: string (nullable = true)
 |-- Preferred_Foot: string (nullable = true)
 |-- Weak_Foot_Rating: string (

In [7]:
fifa.show(10)

+-----------------+--------------------+-------+---------+---------+----------------+-------------+-----------+--------------------+---+------+------+----------+---------+-------------------+------+--------------+-------------+--------------+------------------+---------+-------+--------------+----------------+-----------+------------------------+------------------+------------------------+----------------------+---------------------------+-------------------+-------------------+----------+--------------+-------------+---------------+---------------+-----------------+--------+---------+----------------+-------------+-------+---------+-----+-----------------+-----------+-----------+------------+------------+-------+---------+-------+----------+-------+-------+--------+----------+----------+-------------+-----------+------+---------+---------+-------+---------------+--------------+-----------------+-------------------+------------------+----------------------+-------------------+---------

<font size='7'> <b>2.</b></font><font size='5'> <span style="line-height:1.7">  <b>Nationality</b>(국가)가 <font color='blue'><b>Spain</b></font>인 선수의 <font color='blue'><b>Full_Name, Overall, Age</b></font>를 보이되,<b>Overall 기준으로 내림차순</b>하여 보이시오.</span></font>

In [8]:
fifa.select(col('Nationality'),col('Full_Name'),col('Overall'),col('Age')).where(fifa['Nationality']=='Spain').orderBy(fifa['Overall'].desc()).show()

+-----------+--------------------+-------+---+
|Nationality|           Full_Name|Overall|Age|
+-----------+--------------------+-------+---+
|      Spain|Rodrigo Hernández...|     87| 26|
|      Spain|David De Gea Quin...|     87| 31|
|      Spain|    Thiago Alcântara|     86| 31|
|      Spain| Daniel Parejo Muñoz|     86| 33|
|      Spain|     Aymeric Laporte|     86| 28|
|      Spain|    Jordi Alba Ramos|     85| 33|
|      Spain|Sergio Busquets B...|     85| 33|
|      Spain|   Iago Aspas Juncal|     85| 34|
|      Spain|Gerard Moreno Bal...|     85| 30|
|      Spain|Pedro González López|     85| 19|
|      Spain| Sergio Ramos García|     84| 36|
|      Spain|Daniel Carvajal R...|     84| 30|
|      Spain|Sergio Canales Ma...|     84| 31|
|      Spain|   Iker Muniain Goñi|     84| 29|
|      Spain|Mikel Oyarzabal U...|     84| 25|
|      Spain|Marcos Llorente M...|     84| 27|
|      Spain|José Ángel Esmori...|     83| 25|
|      Spain|Marco Asensio Wil...|     83| 26|
|      Spain|

<font size='7'> <b>3.</b></font><font size='5'> <span style="line-height:1.7"> <font color='blue'> <b>Nationality</b>(국가)별 <b>Overall</b>의 최대와 최소, <b>Age</b>의 평균</font>를 확인할 수 있는 DataFrame을 보이시오. 집계 값을 나타내는 각 <font color='blue'>column명</font>을 적절하게 <font color='blue'>변경</font>하여 보이시오. </br>&nbsp;&nbsp;<font size='3'><b>ex></b> Maximum Overall, Minimum Overall, Average Age</font></span></font>

In [9]:
from pyspark.ml.feature import SQLTransformer
sql_fifa = SQLTransformer(statement="SELECT Nationality, MAX(Overall) AS MAX_Overall, MIN(Overall) AS MIN_Overall, MEAN(Age) AS Mean_Age FROM __THIS__ GROUP BY Nationality")
sql_fifa.transform(fifa).show(10)

+--------------+-----------+-----------+------------------+
|   Nationality|MAX_Overall|MIN_Overall|          Mean_Age|
+--------------+-----------+-----------+------------------+
| Côte d'Ivoire|         84|         53|24.673267326732674|
|        Russia|         79|         57|             28.25|
|      Paraguay|         78|         52|26.103448275862068|
|Chinese Taipei|         69|         64|              30.5|
|       Senegal|         89|         52|25.424242424242426|
|        Sweden|         82|         51|24.869230769230768|
|   Philippines|         74|         57|              27.6|
|          Fiji|         70|         70|              34.0|
|        Turkey|         84|         50| 25.03914590747331|
|          Iraq|         70|         54|              24.0|
+--------------+-----------+-----------+------------------+
only showing top 10 rows



<font size='7'> <b>4.</b></font><font size='5'> <span style="line-height:1.7">  <b>Nationality</b>(국가)별 <b>Club_Position</b>이 <font color='blue'>CM, CDM</font>인 선수들의 <font color='blue'>최대 <b>Overall</b>, 평균 <b>Age</b></font>를 확인할 수 있는 DataFrame을 보이시오.  </span></font>
><span style="line-height:2.0"><font size='4'>각 국가별로 CM과 CDM 각각에 대해 최대 Overall, 평균 Age 총 4개의 값을 보이시오. </span>

In [10]:
# sql_fifa2 = SQLTransformer(statement="SELECT Nationality, Club_Position, MAX(Overall),MEAN(Age) FROM __THIS__ GROUP BY Nationality,Club_Position HAVING")
# sql_fifa2.transform(fifa).show(10)
from pyspark.sql.functions import *
fifa.groupBy('Nationality','Club_Position').agg(max('Overall').alias('MAX_Overall'),mean('Age').alias('MEAN_Age')).where(fifa.Club_Position.isin('CM','CDM')).show(10)

+--------------------+-------------+-----------+------------------+
|         Nationality|Club_Position|MAX_Overall|          MEAN_Age|
+--------------------+-------------+-----------+------------------+
|               Congo|           CM|         71|23.666666666666668|
|               Gabon|           CM|         69|              29.0|
|           Argentina|          CDM|         81|27.157894736842106|
|             Algeria|          CDM|         74|              29.0|
|               Japan|           CM|         78|27.142857142857142|
|         Netherlands|           CM|         87|24.419354838709676|
|             England|           CM|         83| 25.19277108433735|
|       United States|           CM|         80|              24.7|
|Bosnia and Herzeg...|           CM|         74|29.333333333333332|
|             Belgium|          CDM|         73|              23.6|
+--------------------+-------------+-----------+------------------+
only showing top 10 rows



<font size='7'> <b>5.</b></font><font size='5'> <span style="line-height:1.7"> 우리팀에 <b>Overall</b>은 높은편이지만 상대적으로 <b>Value</b>와 5년 간의 <b>Wage</b>이 적은 <font color='blue'> ST position이 가능한 어린 선수를 영입</font>하고자 한다. <font color='blue'><b>Overall</b> 85보다 큰 ST</font> position이 가능한 <font color='blue'>30세 미만</font> 선수 중  <font color='blue'><b>5년간의 Wage와 Value의 합</b>이 가장 적은 <b>3명</b>의 선수</font>의 <b>Nationality, Full Name, Age, Overall</b>를 찾으시오. </font></span></font></br>
><span style="line-height:2.0"><font size='4'>ST position이 가능한 선수란, <b>Positions_Played</b> column에 <font color='blue'>ST가 포함된 선수</font>를 의미한다.</br>
5년간의 Wage와 Value의 합은 <font color='blue'>5 x <b>Wage</b> + <b>Value</b></font>을 의미한다.</font></span>

In [11]:
fifa_po = fifa.select('Nationality','Full_Name','Age','Overall','Wage','Value').where(fifa.Positions_Played.like('%ST%'))
fifa_po = fifa_po.select('Nationality','Full_Name','Age','Overall','Wage','Value').where(fifa_po['Overall']>85)
fifa_po = fifa_po.where(fifa_po['Age']<30)
fifa_po.select("Nationality","Full_Name","Age","Overall",expr('5*(Wage+Value) AS Wage_Value')).orderBy('Wage_Value').show(3)

+-----------+------------------+---+-------+----------+
|Nationality|         Full_Name|Age|Overall|Wage_Value|
+-----------+------------------+---+-------+----------+
|    Belgium|     Romelu Lukaku| 29|     86| 341000000|
|     France|Christopher Nkunku| 24|     86| 470450000|
|  Argentina|  Lautaro Martínez| 24|     86| 493050000|
+-----------+------------------+---+-------+----------+
only showing top 3 rows

